# Lun 12/10 · 11:30 — TP v0.2 : valider et collecter les erreurs

Lun 12/10, 12:00–12:50. Correction commentée à 12:50, TP fini ou non.

## Objectif

Valider chaque serveur du parc, lever une exception métier `InvalidServer` avec un message qui dit quoi corriger, et ranger les serveurs en valides et invalides au lieu de s'arrêter au premier serveur faux. Le résumé de v0.1 ne porte plus que sur les valides.

## Mode d'emploi du notebook

Celui de `tp/c02.ipynb` vaut toujours : cellule d'outils d'abord, `%run fleetcheck.py` après chaque enregistrement, réponse aux **Prédire** et **Justifier** avant d'exécuter. Rendu : *Restart Kernel and Run All Cells*, puis commit et push de `fleetcheck.py` et de ce notebook.

In [ ]:
# Outils du TP : exécuter cette cellule en premier, sans la modifier.
import os
import subprocess
import sys


def check(obtenu, attendu):
    if obtenu == attendu:
        print("OK")
    elif isinstance(attendu, str) and "\n" in attendu:
        print(f"KO, obtenu :\n{obtenu}\nattendu :\n{attendu}")
    else:
        print(f"KO\n  obtenu  : {obtenu!r}\n  attendu : {attendu!r}")


def check_error(fonction, *args, attendu):
    try:
        fonction(*args)
    except Exception as e:
        check(f"{type(e).__name__}: {e}", attendu)
    else:
        print(f"KO, aucune exception levée. Attendu : {attendu}")


def run(*args):
    """Lance fleetcheck.py comme dans un terminal (vu en v0.8 : subprocess)."""
    r = subprocess.run([sys.executable, "fleetcheck.py", *args], capture_output=True,
                       encoding="utf-8", env={**os.environ, "PYTHONUTF8": "1", "NO_COLOR": "1",
                                              "PYTHONWARNDEFAULTENCODING": "1"})
    print(r.stdout, end="")
    print(r.stderr, end="", file=sys.stderr)
    print(f"[code de sortie : {r.returncode}]")
    if "EncodingWarning" in r.stderr:
        print("KO : un open(), read_text() ou subprocess text=True sans encoding= (ligne indiquée dans l'EncodingWarning ci-dessus)")
    return r

## Point de départ

Votre `fleetcheck.py` de v0.1. Sinon, `reference/fleetcheck.py` du tag `v0.1` du dépôt public.

## 1. Le nouveau parc et l'exception — fin 12:05

Remplacez `SERVERS` dans `fleetcheck.py` par la liste ci-dessous. Ne la retapez pas. Les ports sont maintenant des chaînes, comme ils sortiront du CSV en v0.4, et deux lignes fausses sont ajoutées à la fin.

```text
SERVERS = [
    {"name": "web-1", "env": "prod", "ip": "127.0.0.1", "port": "8001", "role": "web"},
    {"name": "web-2", "env": "prod", "ip": "127.0.0.1", "port": "8002", "role": "web"},
    {"name": "api-1", "env": "prod", "ip": "127.0.0.1", "port": "8003", "role": "api"},
    {"name": "api-2", "env": "prod", "ip": "127.0.0.1", "port": "8004", "role": "api"},
    {"name": "worker-1", "env": "prod", "ip": "127.0.0.1", "port": "8005", "role": "worker"},
    {"name": "web-stg-1", "env": "staging", "ip": "127.0.0.1", "port": "8006", "role": "web"},
    {"name": "api-stg-1", "env": "staging", "ip": "127.0.0.1", "port": "8007", "role": "api"},
    {"name": "cache-1", "env": "prod", "ip": "127.0.0.1", "port": "8009", "role": "cache"},
    {"name": "db-1", "env": "prod", "ip": "192.0.2.1", "port": "5432", "role": "db"},
    {"name": "api-3", "env": "prod", "ip": "127.0.0.1", "port": "abc", "role": "api"},
    {"name": "web-3", "env": "prod", "ip": "300.0.0.1", "port": "8010", "role": "web"},
]
```

Écrivez la classe `InvalidServer`, sous-classe d'`Exception`.

Vérification : le parc compte 11 serveurs et `InvalidServer` hérite d'`Exception`.

In [ ]:
%run fleetcheck.py
check((len(SERVERS), issubclass(InvalidServer, Exception)), (11, True))

## 2. `validate(server)` — fin 12:25

`validate(server)` renvoie un **nouveau** dict, identique au serveur reçu sauf `port` converti en `int`. Si le serveur est faux, elle lève `InvalidServer`. Règles, contrôlées dans cet ordre :

| Règle | Message de l'exception |
|---|---|
| Port non entier | `port non entier : 'abc'` |
| Port hors de 1–65535 | `port hors de 1-65535 : 70000` |
| IP invalide (`ipaddress.ip_address`) | `IP invalide : '300.0.0.1'` |
| Champ `name`, `env` ou `role` vide | `champ 'env' vide` |

La règle « champ vide » peut passer en « pour aller plus loin » à 12:25, sur annonce de l'enseignant. La sortie attendue de l'étape 4 n'en dépend pas.

**Prédire** — Quel message porte l'exception levée par ce serveur, faux sur deux règles ?

```text
validate({"name": "x", "env": "prod", "ip": "300.0.0.1", "port": "abc", "role": "web"})
```

In [ ]:
# Ma réponse :

Essayez votre fonction ici, par exemple `validate(SERVERS[10])` :

In [ ]:
%run fleetcheck.py


Vérification : le port du premier serveur devient un entier, le dixième serveur lève `InvalidServer`.

In [ ]:
%run fleetcheck.py
check(validate(SERVERS[0])["port"] + 1, 8002)
check_error(validate, SERVERS[9], attendu="InvalidServer: port non entier : 'abc'")

## 3. `load(raw_servers)` — fin 12:40

`load(raw_servers)` valide chaque serveur et renvoie le tuple `(valides, invalides)`. `valides` est la liste des dicts renvoyés par `validate`. Chaque invalide est un dict `{"name": ..., "error": ...}`, où `error` est le message de l'exception. Un serveur faux n'arrête pas les suivants.

**Prédire** — Avant d'écrire `load` : si le `try` entourait toute la boucle au lieu d'un seul serveur, que vaudrait `[len(x) for x in load(SERVERS)]` ? Quel serveur ne serait jamais examiné ?

In [ ]:
# Ma réponse :

Le parcours attendu, un `try` par serveur :

```mermaid
flowchart LR
  load["load(raw_servers)"] --> suivant["serveur suivant"]
  suivant --> v(["validate(server)"])
  v -- "renvoie un dict" --> ok["ajouté à valides"]
  v -- "lève InvalidServer" --> ko["ajouté à invalides : name, error"]
  ok --> reste(["reste des serveurs ?"])
  ko --> reste
  reste -- oui --> suivant
  reste -- non --> fin["return valides, invalides"]
```

In [ ]:
%run fleetcheck.py


Vérification : 9 valides, 2 invalides.

In [ ]:
%run fleetcheck.py
check([len(x) for x in load(SERVERS)], [9, 2])

## 4. Résumé — fin 12:50

Le résumé de v0.1 porte sur les serveurs valides seulement, puis une ligne par invalide.

Vérification : `run()` lance `fleetcheck.py` comme dans un terminal (`python3 fleetcheck.py`). La sortie doit être exactement :

```text
9 serveurs valides, 2 invalides
par env : prod 7, staging 2
par rôle : web 3, api 3, worker 1, cache 1, db 1
prod/web : web-1, web-2
invalide api-3 : port non entier : 'abc'
invalide web-3 : IP invalide : '300.0.0.1'
```

**Justifier** — Pourquoi `validate` lève une exception au lieu de renvoyer `False` pour un serveur faux ? Que perdrait cette sortie ?

In [ ]:
# Ma réponse :

C'est le critère de fin du TP.

In [ ]:
r = run()
check(r.stdout, """9 serveurs valides, 2 invalides
par env : prod 7, staging 2
par rôle : web 3, api 3, worker 1, cache 1, db 1
prod/web : web-1, web-2
invalide api-3 : port non entier : 'abc'
invalide web-3 : IP invalide : '300.0.0.1'
""")

## Pour aller plus loin

Non corrigé en salle. Aucune ligne fournie n'est concernée : la sortie attendue ne change pas. Testez en ajoutant vos propres lignes fausses dans les cellules ci-dessous, avec `check_error`.

- Refuser un `env` hors de `{"prod", "staging", "dev"}`.
- Refuser deux serveurs de même nom.

In [ ]:
%run fleetcheck.py


## En retard

La version de référence v0.2 est publiée dans `reference/` à la fin de la séance. Copiez-la dans votre dépôt pour démarrer le TP v0.3.